# 🛒 Notebook 1: Amazon Shopping — Class Design

Welcome! In this notebook we design the **classes** for a tiny Amazon-style online store.
Before writing code, good designers sketch the classes and decide *who owns what*.

### Requirements (tiny, on purpose)
1. Browse a **catalog** of products (each with a price and stock).
2. Add items to a **cart**.
3. **Checkout** turns the cart into an **order**.
4. Pay with a choice of **payment methods** (credit card, PayPal, ...).
5. An order moves through states: `PENDING → PAID → SHIPPED` (or `CANCELLED`).
6. Customers get **notified** when the order status changes.


## 🛠️ Setup

```bash
cd 07-object-oriented-design/amazon-shopping
uv sync
```

In VS Code, pick the `.venv` kernel in the kernel picker (top-right of the notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🚫 Bad design first — one "God Class" that does everything

Before we design properly, let's see the **anti-pattern** most beginners reach for.
A single `AmazonStore` class holds products, carts, orders, payment logic, shipping, and
even prints emails. It "works" but every new feature risks breaking unrelated ones.


In [ ]:
# BAD: one class that knows about everything.
class AmazonStore:
    def __init__(self):
        self.products = {}        # sku -> [name, price, stock]
        self.cart = {}            # sku -> qty  (only one global cart!)
        self.orders = []

    def add_product(self, sku, name, price, stock):
        self.products[sku] = [name, price, stock]

    def add_to_cart(self, sku, qty):
        self.cart[sku] = self.cart.get(sku, 0) + qty

    def checkout(self, card_number):
        total = sum(self.products[s][1] * q for s, q in self.cart.items())
        # deduct stock
        for s, q in self.cart.items():
            self.products[s][2] -= q
        # charge card inline
        print(f"charging ${total} to card {card_number[-4:]}")
        # make order
        self.orders.append({"items": dict(self.cart), "total": total, "status": "PAID"})
        self.cart = {}
        # email the customer inline
        print("sending email: your order is confirmed")

store = AmazonStore()
store.add_product("BOOK-1", "Clean Code", 25, 3)
store.add_to_cart("BOOK-1", 2)
store.checkout("4111111111111111")
print(store.orders)


### What's wrong with the god class?

| Problem | Why it hurts |
|---|---|
| 🧩 Only **one cart** — every user shares it | Can't model multiple shoppers |
| 💳 Payment is **hard-coded** to credit cards | Adding PayPal means editing `checkout` |
| 📧 Email is **inline** in `checkout` | Can't swap for SMS; can't test without sending |
| 🧊 Order is a raw **dict** | Typos like `order["stats"]` silently return `None` |
| 🔁 Prices aren't **frozen** at checkout | If a price changes later, old orders change too |
| 🧪 Hard to **test** in isolation | Everything is tangled together |

These are all violations of the **Single Responsibility Principle** (SRP):
> *A class should have one, and only one, reason to change.*


## ✅ Better design — split responsibilities into small classes

We'll split the god class into focused classes. Each one has **one job**.

```
┌───────┐ 1   * ┌─────────┐
│ Store │──────▶│ Product │     Store owns the catalog & runs checkout
└───────┘       └─────────┘
    │
    ▼
┌──────┐ 1   * ┌──────────┐ * 1 ┌─────────┐
│ Cart │──────▶│ CartItem │────▶│ Product │
└──────┘       └──────────┘     └─────────┘
    │ checkout
    ▼
┌───────┐ 1 1 ┌─────────────────┐       ┌─────────────────┐
│ Order │────▶│ PaymentMethod   │◀──────│ CreditCard /    │ (Strategy)
│       │     └─────────────────┘       │ PayPal / ...    │
│       │                                   └─────────────────┘
│       │ notifies
│       ▼
│   ┌────────────────┐    ┌──────────┐
└──▶│ OrderObserver  │◀───│ Emailer  │ (Observer)
    └────────────────┘    └──────────┘
```

### Responsibilities
| Class           | Job                                                                 |
|-----------------|---------------------------------------------------------------------|
| `Product`       | SKU, name, price, stock — just data.                                |
| `Cart`          | Hold `sku -> qty`; `add`, `remove`, `total`.                        |
| `Order`         | A **snapshot** of a cart + a status.                                |
| `OrderStatus`   | Enum of legal states: `PENDING`, `PAID`, `SHIPPED`, `CANCELLED`.    |
| `PaymentMethod` | Interface with `pay(amount)`. Concrete: `CreditCard`, `PayPal`.     |
| `OrderObserver` | Reacts to status changes (email, SMS, analytics).                   |
| `Store`         | Catalog + `checkout` — the **transaction boundary**.                |

### Design patterns we will use
- **Strategy** — swap payment methods without touching `Store.checkout`.
- **Observer** — notify customers when order status changes.
- **State (light)** — enforce legal order transitions.


## Stubs (no logic yet) — just responsibilities

Let's sketch each class with method signatures only. In **Notebook 2** we implement them.


In [ ]:
from enum import Enum
from dataclasses import dataclass
from abc import ABC, abstractmethod

class OrderStatus(Enum):
    PENDING = 1
    PAID = 2
    SHIPPED = 3
    CANCELLED = 4

@dataclass
class Product:
    sku: str
    name: str
    price: float
    stock: int

class PaymentMethod(ABC):
    @abstractmethod
    def pay(self, amount: float) -> bool: ...

class OrderObserver(ABC):
    @abstractmethod
    def on_status_change(self, order, old, new) -> None: ...

# Legal transitions -- used by Order in notebook 2
LEGAL_TRANSITIONS = {
    OrderStatus.PENDING:   {OrderStatus.PAID, OrderStatus.CANCELLED},
    OrderStatus.PAID:      {OrderStatus.SHIPPED, OrderStatus.CANCELLED},
    OrderStatus.SHIPPED:   set(),
    OrderStatus.CANCELLED: set(),
}

print("Legal transitions from PENDING:", [s.name for s in LEGAL_TRANSITIONS[OrderStatus.PENDING]])
print("Legal transitions from SHIPPED:", [s.name for s in LEGAL_TRANSITIONS[OrderStatus.SHIPPED]])


## 🧠 Quick mental check

Before moving on, ask yourself:
1. **Why** does `Order` store a *copy* of the cart lines and total (not a reference)?
   → Because prices and stock in the catalog change over time. An order must never change.
2. **Where** does adding a new payment method (e.g. ApplePay) touch existing code?
   → Only a new subclass of `PaymentMethod`. `Store.checkout` is untouched. (**Open/Closed Principle**.)
3. **Why** is `OrderObserver` a separate class, not a method on `Store`?
   → So we can add/remove notifiers (email, SMS, Slack) without changing checkout logic.

👉 Continue to **Notebook 2** where we implement everything step-by-step,
starting from a broken version and refactoring toward the clean design.
